In [2]:
import torch
import torch.nn as nn
import mlx
import wikipediaapi
torch.__version__
device = "mps"

In [3]:
wiki = wikipediaapi.Wikipedia(
    user_agent="kickoff-football-bot/1.0 (aaravgarg1011@gmail.com)",  # Wikipedia requires this
    language="en",
)
clubs = ["FC Barcelona", "Manchester City F.C.", "Liverpool F.C.", "Real Madrid CF"]
text = ""
for club in clubs:
    page = wiki.page(club)
    if page.exists():
        text += page.text + "\n\n"
        print(f"Got {club}: {len(page.text)} chars")
    else:
        print(f"NOT FOUND: {club}  (check the title)")
with open("football.txt", "w", encoding="utf-8") as f:
    f.write(text)

Got FC Barcelona: 69732 chars
Got Manchester City F.C.: 42439 chars
Got Liverpool F.C.: 40527 chars
Got Real Madrid CF: 79880 chars


In [4]:
with open('football.txt', 'r') as F:
    text = F.read()
print(len(text))
print(text[:500])

232586
Futbol Club Barcelona (Catalan pronunciation: [fudˈbɔl ˈklub bəɾsəˈlonə] ), commonly known as FC Barcelona and colloquially as Barça ([ˈbaɾsə]), is a professional football club based in Barcelona, Catalonia, Spain, that competes in La Liga, the top division of Spanish football.
Founded in 1899 by a group of Swiss, Catalan, German, and English footballers led by Joan Gamper, the club has become a symbol of Catalan culture and Catalanism, hence the motto Més que un club ("More than a club"). Unlik


In [5]:
words = text.split()
print(f"Total tokens: {len(words)}")
print(f"Unique tokens: {len(set(words))}")
print(words[:30])

Total tokens: 38503
Unique tokens: 7350
['Futbol', 'Club', 'Barcelona', '(Catalan', 'pronunciation:', '[fudˈbɔl', 'ˈklub', 'bəɾsəˈlonə]', '),', 'commonly', 'known', 'as', 'FC', 'Barcelona', 'and', 'colloquially', 'as', 'Barça', '([ˈbaɾsə]),', 'is', 'a', 'professional', 'football', 'club', 'based', 'in', 'Barcelona,', 'Catalonia,', 'Spain,', 'that']


In [6]:
vocab = sorted(set(words))
stoi = {w: idx for idx, w in enumerate(vocab)} #String to integer
itos = {idx: w for w, idx in stoi.items()}
vocab_size = len(vocab)

print("stoi value type:", type(list(stoi.values())[0]))
print("stoi sample:", list(stoi.items())[:3])
print("itos sample:", list(itos.items())[:3])

def encode(s): 
    return [stoi[w] for w in s.split()]
def decode(ids):
    return " ".join(itos[i] for i in ids)

t = encode("Barcelona is a football club")
print(f"Encoded: {t}")
print(f"Deconde: {decode(t)}")


stoi value type: <class 'int'>
stoi sample: [('"A', 0), ('"Anfield', 1), ('"At', 2)]
itos sample: [(0, '"A'), (1, '"Anfield'), (2, '"At')]
Encoded: [1265, 5092, 3229, 4658, 3814]
Deconde: Barcelona is a football club


In [ ]:
batch_size = 32     # how many sequences per batch
block_size = 64     # how many words in each sequence

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))     # 32 random start spots
    x = torch.stack([d[i : i + block_size] for i in ix])       # the chunks
    y = torch.stack([d[i+1 : i+block_size+1] for i in ix])     # same chunks, shifted one right
    x, y = x.to(device), y.to(device)
    return x, y

xb, yb = get_batch("train")
print(xb.shape, yb.shape)

torch.Size([32, 64]) torch.Size([32, 64])


In [7]:
data = torch.tensor(encode(text), dtype=torch.long)

n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

print("total tokens:", len(data))
print("train:", len(train_data), " val:", len(val_data))
print("first 20 numbers:", data[:20])

total tokens: 38503
train: 34652  val: 3851
first 20 numbers: tensor([1868, 1520, 1265,  210, 5925, 3224, 7303, 3670,  300, 3875, 5145, 3446,
        1764, 1265, 3373, 3850, 3446, 1279,  241, 5092])


In [8]:
# print("len(text):         ", len(text))
# print("len(words):        ", len(words))
# print("len(vocab):        ", len(vocab))
# print("vocab_size:        ", vocab_size)
# print("len(stoi):         ", len(stoi))
# print("biggest stoi value:", max(stoi.values()))
# print("biggest in data:   ", data.max().item())
# print("model table size:  ", model.embedding.num_embeddings)

In [10]:
class KickOffModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=vocab_size)

    def forward(self, idx):
        logits = self.embedding(idx)
        return logits
        

model = KickOffModel().to(device)
logits = model(xb)
print(logits.shape)
    

NameError: name 'xb' is not defined

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(params=model.parameters(), lr = 1e-3)

B, T, C = logits.shape
loss = loss_fn(logits.view(B*T, C), yb.view(B*T))
print(loss.item())

AcceleratorError: index 17939 is out of bounds for dimension with size 7350

In [ ]:
print("vocab_size:", vocab_size)
print("logits shape:", model(xb).shape)
print("yb max:", yb.max().item())   

vocab_size: 7350
logits shape: torch.Size([32, 64, 7350])


AcceleratorError: index 16550 is out of bounds for dimension with size 7350

In [ ]:
epochs = 1000
for epoch in range(epochs):
    model.train()

    

NameError: name 'model' is not defined